# 10 Facets

This notebook audits `tl.facets`, TorchLens' semantic view layer. A facet is a derived, named view over a captured record: custom recipes can expose domain-specific values, and built-in recipes expose common model semantics such as attention `q`, `k`, and `v` tensors.

We start with a local module so the custom recipe path is deterministic and has no optional dependencies.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
from typing import Any

import torch
from torch import nn
import torchlens as tl

torch.manual_seed(43)


class AuditProjector(nn.Module):
    """Small local module used for a custom facet recipe."""

    def __init__(self) -> None:
        """Initialize the projection stack."""

        super().__init__()
        self.up = nn.Linear(4, 6)
        self.down = nn.Linear(6, 3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Project through a hidden ReLU bottleneck.

        Parameters
        ----------
        x:
            Input tensor with four features.

        Returns
        -------
        torch.Tensor
            Output tensor with three features.
        """

        return self.down(torch.relu(self.up(x)))


class FacetNet(nn.Module):
    """Wrapper with a named custom module."""

    def __init__(self) -> None:
        """Initialize the wrapper."""

        super().__init__()
        self.projector = AuditProjector()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the projector.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Projected tensor.
        """

        return self.projector(x)


model = FacetNet().eval()
x = torch.randn(2, 4)
trace = tl.trace(model, x, layers_to_save="all")
module = trace.modules["projector"]
print(f"module class: {module.class_name}")
print(f"module layers: {[layer.layer_label for layer in module.layers]}")

`tl.facets.register(...)` attaches a recipe to records matching a class name, qualified class name, or predicate. The recipe below targets our local module record and returns three lightweight semantic values.

In [ ]:
import torchlens.semantic.facets as _facet_impl

_facet_registry_snapshot = list(_facet_impl._REGISTRY)
_facet_impl._REGISTRY[:] = [
    recipe
    for recipe in _facet_impl._REGISTRY
    if recipe.public.recipe_name != "audit_projector_recipe"
]


@tl.facets.register(
    class_name="AuditProjector",
    target_scope="module",
    facets=("output_norm", "num_internal_layers", "output_shape"),
)
def audit_projector_recipe(record: Any) -> dict[str, Any]:
    """Return semantic facets for the audit projector.

    Parameters
    ----------
    record:
        TorchLens module record for `AuditProjector`.

    Returns
    -------
    dict[str, Any]
        Derived semantic values.
    """

    return {
        "output_norm": float(record.out.detach().norm().item()),
        "num_internal_layers": len(record.layers),
        "output_shape": tuple(record.out_shape),
    }


view = module.facets
print(f"view type: {type(view).__name__}")
print(f"recipe source: {view.recipe_source}")
print(f"keys: {view.keys()}")

`FacetView` is both dict-like and attribute-like. `keys()` is cheap because recipes declare their facet names before values are computed.

In [ ]:
print(f"attribute access: output_norm={view.output_norm:.6f}")
print(f"subscript access: output_shape={view['output_shape']}")
print(f"has num_internal_layers: {view.has('num_internal_layers')}")
print(f"missing with default: {view.get('not_registered', 'fallback')}")

The registry is inspectable, which is useful when documenting which semantic recipes a project has installed.

In [ ]:
for recipe in tl.facets.list(class_name="Audit*"):
    print(f"recipe={recipe.recipe_name} classes={recipe.class_names} scope={recipe.target_scope}")
print(tl.facets.info("AuditProjector"))

If `transformers` is importable, we instantiate a tiny DistilBERT attention module directly from config and show the built-in attention facets. This avoids model downloads while still using the real Hugging Face attention class.

In [ ]:
try:
    from transformers import DistilBertConfig
    from transformers.models.distilbert.modeling_distilbert import MultiHeadSelfAttention
except Exception as exc:
    print(
        "> NOTE: transformers is not available, so this run skips built-in HF attention "
        "facets. Built-in q/k/v recipes target Hugging Face attention class names."
    )
    print(f"skip reason: {type(exc).__name__}: {exc}")
else:

    class HFAttentionWrapper(nn.Module):
        """Tiny wrapper around a Hugging Face DistilBERT attention module."""

        def __init__(self) -> None:
            """Initialize a tiny attention block from config only."""

            super().__init__()
            config = DistilBertConfig(
                vocab_size=32,
                n_layers=1,
                dim=8,
                hidden_dim=16,
                n_heads=2,
                dropout=0.0,
                attention_dropout=0.0,
            )
            self.attn = MultiHeadSelfAttention(config)

        def forward(self, hidden: torch.Tensor) -> torch.Tensor:
            """Run self-attention and return the tensor output.

            Parameters
            ----------
            hidden:
                Hidden states shaped `(batch, sequence, dim)`.

            Returns
            -------
            torch.Tensor
                Attention output tensor.
            """

            mask = torch.ones(hidden.shape[:2], dtype=torch.bool, device=hidden.device)
            return self.attn(hidden, hidden, hidden, mask, output_attentions=False)[0]

    hf_model = HFAttentionWrapper().eval()
    hf_trace = tl.trace(hf_model, torch.randn(1, 4, 8), layers_to_save="all")
    attn_facets = hf_trace.modules["attn"].facets
    print(f"attention recipe: {attn_facets.recipe_source}")
    print(f"attention keys: {attn_facets.keys()}")
    print(
        f"q/k/v shapes: {tuple(attn_facets.q.shape)} {tuple(attn_facets.k.shape)} {tuple(attn_facets.v.shape)}"
    )
    print(f"head(1).q shape: {tuple(attn_facets.head(1).q.shape)}")

> NOTE: Facets are derived views over captured records. If a recipe expects internal child activations, capture with `layers_to_save="all"` or the required child layers so the recipe has tensors to read.

## Built-In Torch Facets and Discovery
The registry currently includes deterministic torch-only recipes for `Embedding` and `LayerNorm`. Linear and generic MLP family recipes are not exported in this build, while transformer MLP recipes target HF class names. The finder cell below uses registry discovery and then prints actual facet keys from traced modules.

In [ ]:
class BuiltinFacetNet(nn.Module):
    """Torch-only model with modules covered by built-in facet recipes."""

    def __init__(self) -> None:
        """Initialize embedding and normalization modules."""
        super().__init__()
        self.embedding = nn.Embedding(8, 4)
        self.norm = nn.LayerNorm(4)
        self.linear = nn.Linear(4, 4)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """Embed, normalize, and project token ids."""
        return self.linear(self.norm(self.embedding(tokens))).mean(dim=1)


builtin_trace = tl.trace(BuiltinFacetNet().eval(), torch.tensor([[1, 2, 3]]), layers_to_save="all")
for address in ["embedding", "norm", "linear"]:
    module_record = builtin_trace.modules[address]
    facet_view = module_record.facets
    print(
        f"{address}: class={module_record.class_name} recipe={facet_view.recipe_source} keys={facet_view.keys()}"
    )

for query in ["Embedding", "LayerNorm", "Linear", "MLP"]:
    recipes = tl.facets.list(class_name=query)
    if recipes:
        print(f"finder {query}: {[recipe.recipe_name for recipe in recipes]}")
    else:
        print(f"> NOTE: no built-in facet recipe found for {query!r} in this build.")

In [ ]:
sdpa_recipes = [recipe for recipe in tl.facets.list() if "sdpa" in recipe.recipe_name.lower()]
print(f"fused/SDPA recipes: {[recipe.recipe_name for recipe in sdpa_recipes]}")
print(
    "> NOTE: fused SDPA facets are pattern-limited: recipes can expose q/k/v only when the "
    "module class and internal operation pattern match a registered attention recipe. "
    "A raw torch.nn.functional.scaled_dot_product_attention call has no torch-only facet recipe here."
)

## 🔧 Sandbox

Mini-experiment: change `sandbox_facet`, `sandbox_scale`, and `cleanup_registry`. Expected observation: scaling the input changes `output_norm`; cleanup restores the facet registry after the recipe demo.

In [ ]:
sandbox_scale = 1.5
sandbox_facet = "output_norm"
cleanup_registry = True
sandbox_trace = tl.trace(model, x * sandbox_scale, layers_to_save="all")
sandbox_view = sandbox_trace.modules["projector"].facets
baseline_value = view[sandbox_facet]
sandbox_value = sandbox_view[sandbox_facet]
print(f"scale: 1.0 -> {sandbox_scale}")
print(f"{sandbox_facet}: {baseline_value} -> {sandbox_value}")
print(f"facet keys: {sandbox_view.keys()}")
if cleanup_registry:
    _facet_impl._REGISTRY[:] = _facet_registry_snapshot
    print("registry restored after custom recipe demo")